# Loading the required libraries

In [86]:
import pandas as pd 
import numpy as np
import spacy
from spacy import displacy
import networkx as nx
import os
import matplotlib.pyplot as plt
import scipy
import re
import nltk
import re
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
from collections import Counter
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mallika\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Mallika\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Mallika\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


# Loading the page that we scraped in the previous exercise

In [2]:
with open('webpage_source.txt', 'r', errors='ignore') as file: 
   data = file.read().replace( '\n', ' ')

In [46]:
data[0:2000]

'Jump to content Main menu Search Donate Create account Log in Toggle the table of contents Key events of the 20th century 2 languages Article Talk Read Edit View history Tools From Wikipedia, the free encyclopedia The 20th century changed the world in unprecedented ways. The World Wars sparked tension between countries and led to the creation of atomic bombs, the Cold War led to the space race and the creation of space-based rockets, and the World Wide Web was created. These advancements have played a significant role in citizens\' lives and shaped the 21st century into what it is today. Historic events in the 20th century[edit] World at the beginning of the century[edit] Main article: Edwardian era Map of colonial and land-based empires throughout the world in 1914 The new beginning of the 20th century marked significant changes. The 1900s saw the decade herald a series of inventions, including the automobile, airplane and radio broadcasting. 1914 saw the completion of the Panama Can

# Evaluating the text

In [41]:
from nltk.tokenize import word_tokenize
tokenized_word = word_tokenize(data)
#print(tokenized_word) 

In [69]:
from nltk.probability import FreqDist
dist_words = FreqDist(tokenized_word)
print(dist_words)

<FreqDist with 4838 samples and 22336 outcomes>


In [71]:
type(dist_words)

nltk.probability.FreqDist

In [74]:
pd.options.display.max_rows = None

In [79]:
df = pd.DataFrame(list(dist_words.items()), columns=['word', 'frequency'])

In [83]:
df.sort_values(by = 'frequency', ascending = False).head(1000)

,word,frequency
39,.,1642
12,the,1134
31,",",980
14,of,620
46,and,412
1,to,327
34,The,320
79,[,309
81,],309
10,in,307


### Observation : Exploring the text data shows it does have punctuation. The country names appear to be matching what is there in the countiries file that was provided in the previous exercise.

In [65]:
# plt.figure(figsize=(8, 3))
# dist_words.plot(20,cumulative = False)
# plt.show()

In [55]:
# filtered_words = [] # creates an empty list
# for word in tokenized_word:
#       if word not in stopwords.words():
#            filtered_words.append(word)

In [58]:
# dist_words_filter = FreqDist(filtered_words)
# print (dist_words_filter)

<FreqDist with 4575 samples and 16758 outcomes>


In [64]:
# dist_words_filter.most_common(50)

# Creating the NER object

In [5]:
# Download English module

!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------ --------------------------------- 2.1/12.8 MB 10.1 MB/s eta 0:00:02
     ------------- -------------------------- 4.2/12.8 MB 10.2 MB/s eta 0:00:01
     --------------------- ------------------ 6.8/12.8 MB 10.9 MB/s eta 0:00:01
     --------------------------------- ----- 11.0/12.8 MB 13.2 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 12.9 MB/s  0:00:01
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [6]:
NER = spacy.load("en_core_web_sm")

In [7]:
book = NER(data)

In [84]:
# Visualize identified entities

displacy.render(book[273:1000], style = "ent", jupyter = True)

# Splitting the sentence entities from the NER object

In [9]:
df_sentences = [] # empty shell to store results

# Loop through sentences, get entity list for each sentence
for sent in book.sents:
    entity_list = [ent.text for ent in sent.ents]
    df_sentences.append({"sentence": sent, "entities": entity_list})
    
df_sentences = pd.DataFrame(df_sentences)

In [10]:
df_sentences.head(10)

,sentence,entities
0,"(Jump, to, content, Main, menu, Search, Donate...","[Main, Search Donate Create, Key, the 20th cen..."
1,"(The, World, Wars, sparked, tension, between, ...","[The World Wars, the Cold War, the World Wide ..."
2,"(These, advancements, have, played, a, signifi...","[the 21st century, today]"
3,"(Historic, events, in, the, 20th, century[edit...","[20th, 1914]"
4,"(The, new, beginning, of, the, 20th, century, ...",[the 20th century]
5,"(The, 1900s, saw, the, decade, herald, a, seri...","[The 1900s, the decade]"
6,"(1914, saw, the, completion, of, the, Panama, ...","[1914, the Panama Canal]"
7,"(The, scramble, for, Africa, continued, in, th...","[Africa, the 1900s]"
8,"(The, atrocities, in, the, Congo, Free, State,...",[the Congo Free State]
9,"(From, 1914, to, 1918, the, First, World, War,...","[1914 to 1918, the First World War]"


# Filtering the entities to get the countries

In [12]:
countries_list = pd.read_csv('countries_list_20th_century_1.5.csv')

In [13]:
cleaned_list = [row.strip() for row in countries_list['country_name']]

In [15]:
country_df = pd.DataFrame(cleaned_list, columns = ['country_name'])

In [18]:
country_df.head()

,country_name
0,Afghanistan
1,Albania
2,Algeria
3,Andorra
4,Angola


In [16]:
# Function to filter out entities not of interest

def filter_entity(ent_list, character_df):
    return [ent for ent in ent_list 
            if ent in list(country_df['country_name'])]

In [19]:
# Check

filter_entity(["Afghanistan", "CF", "2"], country_df)

['Afghanistan']

In [20]:
df_sentences['character_entities'] = df_sentences['entities'].apply(lambda x: filter_entity(x, country_df))

In [21]:
df_sentences['character_entities'].head(20)

0                    []
1                    []
2                    []
3                    []
4                    []
5                    []
6                    []
7                    []
8                    []
9                    []
10                   []
11                   []
12    [France, Austria]
13             [Russia]
14    [Germany, Russia]
15            [Germany]
16            [Germany]
17                   []
18                   []
19                   []
Name: character_entities, dtype: object

In [22]:
# Filter out sentences that don't have any character entities

df_sentences_filtered = df_sentences[df_sentences['character_entities'].map(len) > 0]

In [23]:
df_sentences_filtered.tail(10)

,sentence,entities,character_entities
1113,"("", The, forgotten, violence, that, helped, In...",[India],[India]
1118,"("", Indian, Independence, Day, :, everything, ...","[Indian Independence Day, Partition between In...",[Pakistan]
1127,"(^, "", The, Philippines, ,, 1898â€“1946, |, US...","[Philippines, 1898â€“1946, US House of Represe...",[Philippines]
1157,"("", Colonial, Cartographies, ,, Postcolonial, ...","[Colonial Cartographies, Postcolonial Borders,...",[Afghanistan]
1197,"(The, Moldovans, :, Romania, ,, Russia, ,, and...","[Moldovans, Romania, Russia, the Politics of C...","[Romania, Russia]"
1260,"("", Selling, ', Operation, Passage, to, Freedo...","[Thomas Dooley, the Religious Overtones of Ear...",[Vietnam]
1294,"("", Stuck, in, Endless, Preliminaries, :, Viet...","[Stuck in Endless Preliminaries, Vietnam, the ...",[Vietnam]
1567,"("", Anti, -, American, Behavior, in, the, Midd...","[Anti-American Behavior, the Middle East, a Fi...",[Lebanon]
1573,"(The, Rise, of, China, and, India, :, A, New, ...","[India, New Asian]",[India]
1574,"(Singapore, :, World, Scientific, ., doi:10.11...","[Singapore, World Scientific]",[Singapore]


# Creating the relationship dataframe

In [24]:
# Defining relationships 

# window size = 5 : this defines how many sentences will be looked at simultaneously 
relationships = [] # create an empty list

for i in range(df_sentences_filtered.index[-1]):
    end_i = min(i+5, df_sentences_filtered.index[-1])
    char_list = sum((df_sentences_filtered.loc[i: end_i].character_entities), [])
    
    # Remove duplicated characters that are next to each other
    char_unique = [char_list[i] for i in range(len(char_list)) 
                   if (i==0) or char_list[i] != char_list[i-1]]
    
    if len(char_unique) > 1:
        for idx, a in enumerate(char_unique[:-1]):
            b = char_unique[idx + 1]
            relationships.append({"source": a, "target": b})

In [25]:
relationship_df = pd.DataFrame(relationships)

In [26]:
relationship_df

,source,target
0,France,Austria
1,France,Austria
2,Austria,Russia
3,France,Austria
4,Austria,Russia
...,...,...
566,India,Singapore
567,India,Singapore
568,India,Singapore
569,India,Singapore


In [27]:
# Sort the cases with a->b and b->a

relationship_df = pd.DataFrame(np.sort(relationship_df.values, axis = 1), columns = relationship_df.columns)
relationship_df.head(5)

,source,target
0,Austria,France
1,Austria,France
2,Austria,Russia
3,Austria,France
4,Austria,Russia


In [28]:
# Summarize the interactions

relationship_df["value"] = 1
relationship_df = relationship_df.groupby(["source","target"], sort=False, as_index=False).sum()

In [30]:
relationship_df.head(10)

,source,target,value
0,Austria,France,6
1,Austria,Russia,5
2,Germany,Russia,21
3,Austria,Germany,13
4,Germany,Italy,25
5,Germany,Spain,2
6,France,Poland,15
7,France,Germany,17
8,Germany,Poland,23
9,Estonia,Germany,5


# Saving the relationship file

In [32]:
relationship_df.to_csv('country_relationship.csv', index= False)